# 🚀 Module 4: Pretraining — Next-Token Prediction

In this notebook, we train MiniMind from scratch using the language modeling objective.

**What you'll learn:**
- The pretraining objective: next-token prediction (causal language modeling)
- How the training loop works: forward pass, loss, backward pass, optimizer step
- Mixed precision training (bfloat16/float16) for GPU efficiency
- Gradient accumulation for larger effective batch sizes
- Cosine learning rate scheduling
- How to monitor and visualize training progress

**Key insight**: Pretraining on "predict the next word" forces the model to learn grammar, facts, reasoning — everything compressible about language.

> ⏱️ **Expected training time**: ~5-15 minutes on a T4 GPU for 1 epoch of the mini dataset

In [ ]:
# Install dependencies and set up environment
!pip install -q modelscope transformers==4.48.0 torch matplotlib tqdm
import subprocess, os

if not os.path.exists('/content/minimind-colab'):
    subprocess.run(['git', 'clone', 'https://github.com/Boyu-Zhang-UOI/minimind-colab', '/content/minimind-colab'])
    os.chdir('/content/minimind-colab')
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])

import sys
sys.path.insert(0, '/content/minimind-colab')

!nvidia-smi
import gc

## 💾 Optional: Google Drive Mount

Mount Google Drive to persist your trained model checkpoint across Colab sessions.

In [ ]:
# Uncomment to mount Google Drive and save checkpoints persistently
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = '/content/drive/MyDrive/minimind-checkpoints'

print("Google Drive mount skipped (uncomment above to enable)")

## 📥 Download Pretrain Dataset

We'll use the mini pretrain dataset (~50K samples) for quick experimentation.
The full dataset has millions of samples and takes much longer to train.

In [ ]:
import os
os.makedirs('/content/minimind-colab/dataset', exist_ok=True)

!modelscope download --dataset gongjy/minimind_dataset \
    pretrain_t2t_mini.jsonl \
    --local_dir /content/minimind-colab/dataset

# Verify download
import os
path = '/content/minimind-colab/dataset/pretrain_t2t_mini.jsonl'
if os.path.exists(path):
    size_mb = os.path.getsize(path) / 1e6
    line_count = sum(1 for _ in open(path))
    print(f"✅ Dataset ready: {size_mb:.1f}MB, {line_count:,} samples")

## ⚙️ Training Configuration

Let's configure our training run. We'll use `argparse.Namespace` to organize hyperparameters (this mirrors how the `trainer/train_pretrain.py` script works with command-line args).

**Key hyperparameters:**
| Parameter | Value | Why |
|-----------|-------|-----|
| `batch_size` | 16 | Fits in T4 GPU memory with seq_len=256 |
| `accumulation_steps` | 8 | Effective batch = 16×8 = 128 |
| `learning_rate` | 5e-4 | Standard for transformer pretraining |
| `max_seq_len` | 256 | Shorter = faster training, less memory |
| `dtype` | bfloat16 | Better dynamic range than float16 on T4 |

In [ ]:
import argparse
import os
import torch

args = argparse.Namespace(
    # Paths
    save_dir='/content/minimind-colab/out',
    save_weight='pretrain',
    data_path='/content/minimind-colab/dataset/pretrain_t2t_mini.jsonl',
    # Model architecture
    hidden_size=768,
    num_hidden_layers=8,
    use_moe=0,
    # Training
    epochs=1,
    batch_size=16,
    learning_rate=5e-4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16',
    num_workers=2,
    accumulation_steps=8,
    grad_clip=1.0,
    log_interval=50,
    save_interval=500,
    max_seq_len=256,
    from_weight='none',
)

os.makedirs(args.save_dir, exist_ok=True)
print("Training configuration:")
print(f"  Device:            {args.device}")
print(f"  Batch size:        {args.batch_size} (effective: {args.batch_size * args.accumulation_steps})")
print(f"  Learning rate:     {args.learning_rate}")
print(f"  Max seq len:       {args.max_seq_len}")
print(f"  Precision:         {args.dtype}")
print(f"  Epochs:            {args.epochs}")
print(f"  Grad accumulation: {args.accumulation_steps} steps")

## 🏗️ Initialize Model & Data

We'll initialize:
1. **Model**: Fresh MiniMind weights (random initialization)
2. **Tokenizer**: Pre-trained BPE tokenizer (always reused, never trained from scratch here)
3. **Dataset**: Wrapped in `PretrainDataset` with `SkipBatchSampler` for resumable training
4. **Optimizer**: AdamW with cosine learning rate schedule

In [ ]:
import sys
import torch
from contextlib import nullcontext
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

sys.path.insert(0, '/content/minimind-colab')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset
from trainer.trainer_utils import setup_seed, get_model_params, SkipBatchSampler

# Set random seed for reproducibility
setup_seed(42)

# Create model config and initialize model
lm_config = MiniMindConfig(
    hidden_size=args.hidden_size,
    num_hidden_layers=args.num_hidden_layers,
    use_moe=bool(args.use_moe)
)
tokenizer = AutoTokenizer.from_pretrained('/content/minimind-colab/model')
model = MiniMindForCausalLM(lm_config).to(args.device)
get_model_params(model, lm_config)

# Create dataset
train_ds = PretrainDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
print(f"\nDataset: {len(train_ds):,} samples")

# Optimizer and mixed precision setup
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
dtype = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
device_type = 'cuda' if 'cuda' in args.device else 'cpu'
autocast_ctx = (nullcontext() if device_type == 'cpu'
                else torch.cuda.amp.autocast(dtype=dtype))
scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

print(f"\nOptimizer: AdamW, dtype: {dtype}")

## 📉 Training Loop with Live Loss Plotting

The training loop implements:

1. **Forward pass**: Model predicts next tokens, compute cross-entropy loss
2. **Gradient accumulation**: Accumulate gradients over `accumulation_steps` batches
3. **Gradient clipping**: Prevent exploding gradients (`grad_clip=1.0`)
4. **Cosine LR schedule**: Learning rate decays from `lr` to `lr/10` over training
5. **Mixed precision**: Use bfloat16 for forward pass, float32 for optimizer

The `get_lr()` function implements cosine annealing:
$$lr(t) = lr_{\min} + \frac{1}{2}(lr_{\max} - lr_{\min})\left(1 + \cos\left(\frac{\pi t}{T}\right)\right)$$

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import clear_output
from tqdm.notebook import tqdm
import time
import gc
from trainer.trainer_utils import get_lr

matplotlib.rcParams['figure.figsize'] = (12, 4)

loss_history = []
step_history = []

def train_with_plot(epochs=1):
    model.train()
    global_step = 0

    for epoch in range(epochs):
        indices = torch.randperm(len(train_ds)).tolist()
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader = DataLoader(
            train_ds,
            batch_sampler=batch_sampler,
            num_workers=args.num_workers,
            pin_memory=True
        )
        iters = len(loader)
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, (input_ids, labels) in enumerate(pbar, start=1):
            global_step += 1
            input_ids = input_ids.to(args.device)
            labels = labels.to(args.device)

            # Cosine learning rate schedule
            lr = get_lr(epoch * iters + step, epochs * iters, args.learning_rate)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

            # Forward pass with mixed precision
            with autocast_ctx:
                res = model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / args.accumulation_steps

            scaler.scale(loss).backward()

            # Gradient accumulation: only step every N batches
            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            current_loss = loss.item() * args.accumulation_steps
            loss_history.append(current_loss)
            step_history.append(global_step)
            pbar.set_postfix({'loss': f'{current_loss:.4f}', 'lr': f'{lr:.2e}'})

            # Live loss plot every log_interval steps
            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(12, 4))
                ax.plot(step_history, loss_history, 'b-', alpha=0.4, linewidth=0.8, label='Raw')
                if len(loss_history) > 20:
                    window = 20
                    smoothed = [
                        sum(loss_history[max(0, i-window):i+1]) / min(i+1, window)
                        for i in range(len(loss_history))
                    ]
                    ax.plot(step_history, smoothed, 'r-', linewidth=2, label='Smoothed')
                ax.set_xlabel('Step')
                ax.set_ylabel('Loss')
                ax.set_title(f'Pretraining Loss (Epoch {epoch+1}/{epochs}, Step {global_step})')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

        # Save checkpoint at end of epoch
        model.eval()
        moe_suffix = '_moe' if lm_config.use_moe else ''
        ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}{moe_suffix}.pth'
        state_dict = model.state_dict()
        torch.save({k: v.half().cpu() for k, v in state_dict.items()}, ckp)
        print(f"✅ Checkpoint saved: {ckp}")
        model.train()

    return loss_history

print("✅ Training function ready!")
print(f"Starting training: {args.epochs} epoch(s)")
print(f"Estimated steps per epoch: ~{len(train_ds) // args.batch_size}")

In [ ]:
# Start training!
loss_history = train_with_plot(epochs=args.epochs)
print(f"\n🎉 Training complete!")
print(f"   Steps: {len(loss_history)}")
print(f"   Initial loss: {loss_history[0]:.4f}")
print(f"   Final loss:   {loss_history[-1]:.4f}")

In [ ]:
# Final training curve with smoothing
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(step_history, loss_history, 'b-', alpha=0.4, linewidth=0.8, label='Raw loss')

window = 50
if len(loss_history) > window:
    smoothed = [
        sum(loss_history[max(0, i-window):i+1]) / min(i+1, window)
        for i in range(len(loss_history))
    ]
    ax.plot(step_history, smoothed, 'r-', linewidth=2, label=f'Smoothed (window={window})')

ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('MiniMind Pretraining Loss Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/minimind-colab/pretrain_loss.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss:   {loss_history[-1]:.4f}")
print(f"Improvement:  {(loss_history[0] - loss_history[-1]):.4f} ({(loss_history[0] - loss_history[-1])/loss_history[0]*100:.1f}%)")
print(f"\nLoss curve saved to /content/minimind-colab/pretrain_loss.png")

## 🧪 Test the Pretrained Model

A **pretrained** model (no SFT yet) behaves differently from a chatbot:
- It continues text (word completion), not answers questions
- It may generate incoherent or off-topic text for short prompts
- This is normal! SFT will teach it to follow instructions

Let's test it with simple text completion prompts.

In [ ]:
model.eval()

test_prompts = [
    "北京是中国的",
    "人工智能是",
    "Python语言的特点是"
]

print("=== Pretrained Model (Text Completion Mode) ===\n")
for prompt in test_prompts:
    input_text = tokenizer.bos_token + prompt
    inputs = tokenizer(input_text, return_tensors="pt").to(args.device)
    
    with torch.no_grad():
        generated = model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    output = tokenizer.decode(generated[0], skip_special_tokens=True)
    print(f"Prompt:  {prompt!r}")
    print(f"Output:  {output!r}")
    print()

In [ ]:
# Clean up GPU memory
import gc
torch.cuda.empty_cache()
gc.collect()
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")
print("Memory cleared!")

## 📝 Student Exercise

1. **Experiment with sequence length**: Change `max_seq_len` to 128 or 512. How does it affect:
   - Training speed (steps per second)?
   - Memory usage?
   - Loss convergence?

2. **Experiment with batch size**: Try `batch_size=8` with `accumulation_steps=16`. The effective batch size stays the same — does the training look different?

3. **Learning rate sensitivity**: Try `learning_rate=1e-3` (too high) and `learning_rate=1e-5` (too low). What do the loss curves look like?

4. **Save and reload**: After training, load your checkpoint:
```python
ckp = '/content/minimind-colab/out/pretrain_768.pth'
state_dict = torch.load(ckp, map_location='cpu')
model.load_state_dict({k: v.float() for k, v in state_dict.items()})
```

## 💡 Key Observation: Pretrain vs SFT

After pretraining, the model can **complete text** but cannot **follow instructions**.

| Behavior | Pretrained Model | SFT Model |
|----------|-----------------|-----------|
| Input: "What is AI?" | Continues with more text about AI | Answers: "AI is artificial intelligence..." |
| Input: "北京是..." | Continues: "...中国的首都" | Only generates if prompted correctly |
| Instruction following | ❌ No | ✅ Yes |
| World knowledge | ✅ Basic | ✅ Basic |

**The next step** is Supervised Fine-Tuning (SFT), where we train on instruction-response pairs using `train_full_sft.py`. This teaches the model to follow the chat template format and respond helpfully.

The checkpoint you saved (`pretrain_768.pth`) will be the starting point for SFT training!